# OpenDART 공시 원문 + IR PDF × Gemini AI 정성 리스크 감시기 (Google Colab)

**목적:** 재무 숫자로는 잡히지 않는 **정성적 투자 리스크(경고 신호 ⑤·⑥·⑦)**를 **공식 OpenDART 공시 원문 + Google Drive 업로드 IR PDF**에서 자동 추출해, **Google AI Studio Gemini API**로 심사역 수준의 정성 리포트를 자동 생성합니다.

### 무엇을 자동으로 판독하나요?
- **신호 ⑤ 배당 변동 사유 분류:** 공시 원문을 읽어 `없음 / 업황성 / 재배치성(자사주 소각 전환) / 규제성` 자동 판정
- **신호 ⑥ 주주환원 정책 추적:** 기업가치 제고 계획(밸류업)·배당성향 목표·자사주 소각 약속이 `상향 / 유지 / 후퇴` 중 어디인지 판독
- **신호 ⑦ 자본배분 비효율 감시:** 순이익 1년치를 넘는 대형 M&A, 계열사 지원·합병, 물적분할, 5% 이상 유상증자·CB/BW 발행 징후 탐지
- **할루시네이션(지어내기) 차단:** 모든 판정에 **공시 접수번호(`rcept_no`) 또는 업로드 파일명 + 원문 인용 문장 + DART 원문 바로가기 링크**를 강제 첨부합니다.

### 사전 준비 (Colab 좌측 🔑 Secrets에 무료 키 2개 등록)
1. **`DART_API_KEY`**: OpenDART 공식 무료 API 키 *(기존 스크리너와 동일)*
2. **`GEMINI_API_KEY`**: [Google AI Studio](https://aistudio.google.com/apikey)에서 무료 발급한 Gemini API 키 *(미입력 시 AI 판독 없이 원문 발췌 리포트만 생성하는 Dry-Run 모드로 동작)*

### IR 자료 Google Drive 업로드 폴더 (자동 생성!)
- 첫 실행 시 Google Drive에 **`MyDrive/Colab Notebooks/dividend_qualitative/ir_uploads/`** 폴더와 종목별 하위 폴더가 자동 생성됩니다.
- 원하는 **기업 IR 발표자료 PDF, 밸류업 자료, 주총 참고서류, 메모(.txt/.md)**를 해당 종목 폴더(예: `ir_uploads/005380_현대차/`)에 넣어두기만 하면 다음 실행 때 DART 공시 원문과 함께 자동으로 읽어 Gemini가 통합 분석합니다.
- *(추가)* `robots.txt`를 준수하는 검증된 공식 공개 IR 사이트(삼성전자·KB금융)는 `AUTO_FETCH_PUBLIC_IR=True`일 때 최신 실적발표 PDF를 합법적으로 자동 수집합니다.


In [ ]:
# ==============================================================================
# 1. 환경 설정 · 한글 폰트 · pypdf · PDF 엔진 설치
# ==============================================================================
import os, sys, re, io, gzip, json, time, base64, zipfile, getpass, warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta, date
from html import escape, unescape
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import xml.etree.ElementTree as ET
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트, pypdf, weasyprint 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pypdf", "weasyprint", "pyyaml"], check=False)

import pandas as pd
import requests

try:
    import yaml
except ImportError:
    yaml = None

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

try:
    from IPython.display import display
except ImportError:
    display = print

def load_secret_key(env_name: str, required: bool = True, prompt_label: str = "") -> str:
    key = ""
    secret_note = ""
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get(env_name) or "").strip()
        except Exception as e:
            ename = type(e).__name__
            if "NotebookAccessError" in ename:
                secret_note = f"Colab 좌측 🔑(보안 비밀) 탭에서 '{env_name}'의 [노트북 액세스] 스위치를 ON으로 켜주세요"
            elif "SecretNotFoundError" in ename:
                secret_note = f"Colab 좌측 🔑(보안 비밀) 탭에 '{env_name}' 미등록"
            elif "TimeoutException" in ename:
                secret_note = f"'{env_name}' 보안 비밀 권한 팝업 시간 초과 (🔑 탭에서 [노트북 액세스] ON 권장)"
            else:
                secret_note = f"{ename}: {e}"
            key = ""
    if not key:
        key = os.environ.get(env_name, "").strip()
    if not key and secret_note:
        print(f"[안내] {env_name}: {secret_note}")
    if not key and required and IN_COLAB:
        try:
            key = getpass.getpass(f"{prompt_label or env_name}를 입력하세요 (빈칸 시 건너뜀): ").strip()
        except Exception:
            key = ""
    return key

DART_API_KEY = load_secret_key("DART_API_KEY", required=True, prompt_label="OpenDART API Key(DART_API_KEY)")
GEMINI_API_KEY = load_secret_key("GEMINI_API_KEY", required=False, prompt_label="Google AI Studio API Key(GEMINI_API_KEY)")

print(f"[키 로드] DART_API_KEY={'OK ('+str(len(DART_API_KEY))+'자)' if DART_API_KEY else '미설정'} | "
      f"GEMINI_API_KEY={'OK ('+str(len(GEMINI_API_KEY))+'자)' if GEMINI_API_KEY else '미설정(원문 발췌 Dry-Run)'} | pypdf={'OK' if PdfReader else '미설치'}")

In [ ]:
# ==============================================================================
# 2. 감시 대상 설정 · Google Drive IR 업로드 폴더 자동 생성 · 비공개 포트폴리오 연동
# ==============================================================================
QUAL_CONFIG = {
    # 분석 기간 & 공시 수 제한 (무료 API 속도·토큰 최적화)
    "LOOKBACK_DAYS": 365,            # 최근 1년 공시 감시
    "MAX_DISCLOSURES_PER_STOCK": 8,  # 종목당 원문을 읽을 핵심 공시 최대 건수
    "MAX_CHARS_PER_DOC": 6500,       # 공시/IR 1건당 발췌 최대 글자 수
    "MAX_TOTAL_CONTEXT_CHARS": 36000,# 종목당 Gemini에 넘길 최대 총 글자 수

    # IR 자료 수집 설정
    "AUTO_FETCH_PUBLIC_IR": True,    # robots.txt 준수 검증된 공개 공식 IR PDF 자동 수집 (삼성전자·KB금융)
    "CREATE_STOCK_SUBFOLDERS": True, # Drive ir_uploads/ 아래 종목별 업로드 폴더 자동 생성

    # Gemini AI Studio 모델 설정 (크레딧 0원 Free Tier 공식 지원 최신 모델, 실패 시 자동 폴백)
    "GEMINI_MODELS": ["gemini-4-hydrogen", "gemini-3.8-flash", "gemini-3.5-flash-lite"],
    "GEMINI_SLEEP_SEC": 4.5,         # 무료 티어 분당 호출 제한(RPM) 준수 대기 시간(초)
    "FORCE_REANALYZE": False,        # True: 이번 달 완료된 종목도 Gemini 재분석

    # Drive에 my_kr_portfolio.yaml이 없을 때 사용할 기본 예시 종목 (공개 대표주)
    "DEFAULT_SAMPLE_TICKERS": ["005930", "005380", "105560", "000270", "030190"],

    # 추가로 함께 감시하고 싶은 종목 티커 (예: 스크리너 신규 후보 ["030000", "005180"])
    "EXTRA_TICKERS": [],

    # Google Drive 경로
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_qualitative",
    "PORTFOLIO_YAML": "/content/drive/MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml",
    "ALPHA_LAB_CORP_CACHE": "/content/drive/MyDrive/Colab Notebooks/dividend_alpha_lab/ckpt_corp_codes.json.gz",
    "CACHE_DIR": "./_cache_qualitative",
}

CACHE_DIR = Path(QUAL_CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
IR_UPLOAD_DIR = CACHE_DIR / "ir_uploads"
TARGET_TICKERS = []
TARGET_ROLE = {}  # ticker -> "보유중" | "관심" | "추가"

if IN_COLAB and QUAL_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(QUAL_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        IR_UPLOAD_DIR = DRIVE_ROOT / "ir_uploads"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        IR_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 작업 폴더: {DRIVE_ROOT}")
        print(f"[IR 업로드 폴더] {IR_UPLOAD_DIR}")

        # 비공개 my_kr_portfolio.yaml에서 내 보유주 + 관심주 자동 로드 (ETF/해외 제외, 우선주 -> 보통주 매핑)
        pf_p = Path(QUAL_CONFIG["PORTFOLIO_YAML"])
        if yaml is not None and pf_p.exists():
            pf = yaml.safe_load(pf_p.read_text(encoding="utf-8")) or {}
            for tk, info in (pf.get("overrides") or {}).items():
                if not isinstance(info, dict) or info.get("etf"):
                    continue
                raw_tk = str(info.get("common_ticker") or tk).strip()
                if not raw_tk.isdigit():
                    continue
                ctk = raw_tk.zfill(6)
                if ctk not in TARGET_ROLE:
                    TARGET_TICKERS.append(ctk)
                    TARGET_ROLE[ctk] = "보유중"
            for w_item in (pf.get("watchlist") or []):
                raw_w = str(w_item.get("ticker") if isinstance(w_item, dict) else w_item).strip()
                if not raw_w.isdigit():
                    continue
                wtk = raw_w.zfill(6)
                if wtk not in TARGET_ROLE:
                    TARGET_TICKERS.append(wtk)
                    TARGET_ROLE[wtk] = "관심"
            print(f"[비공개 포트폴리오 연동] 보유/관심 {len(TARGET_TICKERS)}종목 자동 로드 완료!")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 임시 폴더 사용 ({type(e).__name__}: {e})")
else:
    IR_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if not TARGET_TICKERS:
    for tk in QUAL_CONFIG["DEFAULT_SAMPLE_TICKERS"]:
        TARGET_TICKERS.append(tk)
        TARGET_ROLE[tk] = "샘플"

for tk in QUAL_CONFIG["EXTRA_TICKERS"]:
    tk = str(tk).zfill(6)
    if tk not in TARGET_ROLE:
        TARGET_TICKERS.append(tk)
        TARGET_ROLE[tk] = "추가"

# 업로드 안내 파일 자동 생성
readme_ir = IR_UPLOAD_DIR / "README_IR_업로드_안내.txt"
if not readme_ir.exists():
    readme_ir.write_text(
        "=== IR / 리포트 PDF 업로드 방법 ===\n"
        "1. 아래 종목별 폴더(예: 005380_현대차/) 안에 PDF · TXT · MD 파일을 올려두세요.\n"
        "2. 또는 이 폴더(ir_uploads/)에 바로 파일명 앞에 티커나 종목명을 붙여 올려도 자동 인식됩니다.\n"
        "   예) 005380_2026_3Q_IR.pdf, 현대차_밸류업계획.pdf\n"
        "3. 업로드 후 colab_qualitative_ai_monitor.ipynb 를 실행하면 OpenDART 공시 원문과 함께\n"
        "   Gemini AI가 자동으로 읽고 정성 리스크(신호 ⑤⑥⑦)를 통합 분석합니다.\n",
        encoding="utf-8",
    )

print(f"[감시 대상] 총 {len(TARGET_TICKERS)}종목: {[(tk, TARGET_ROLE[tk]) for tk in TARGET_TICKERS]}")

In [ ]:
# ==============================================================================
# 3. OpenDART 고유번호 매핑 + 공시 원문(document.xml) 수집기 + IR PDF 파서
# ==============================================================================
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-qualitative-monitor)"}
DART_BASE = "https://opendart.fss.or.kr/api"
KST = timezone(timedelta(hours=9))
today_dt = datetime.now(KST).date()
as_of_str = today_dt.strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")

def _parse_dart_error_xml(content: bytes) -> str:
    try:
        root = ET.fromstring(content)
        st = (root.findtext("status") or "").strip()
        msg = (root.findtext("message") or "").strip()
        if st or msg:
            return f"status={st}: {msg}"
    except Exception:
        pass
    return content[:160].decode("utf-8", errors="ignore").strip()

def load_corp_map() -> dict[str, dict]:
    """ticker(6자리) -> {'corp_code': ..., 'corp_name': ...}"""
    ckpt_paths = [
        (DRIVE_ROOT / "ckpt_corp_meta.json.gz") if DRIVE_ROOT else None,
        CACHE_DIR / "ckpt_corp_meta.json.gz",
    ]
    for p in ckpt_paths:
        if p and p.exists() and p.stat().st_size > 1000:
            try:
                return json.loads(gzip.decompress(p.read_bytes()).decode("utf-8"))
            except Exception:
                pass

    mapping = {}
    if DART_API_KEY:
        try:
            r = requests.get(f"{DART_BASE}/corpCode.xml", params={"crtfc_key": DART_API_KEY}, headers=_HEADERS, timeout=90)
            r.raise_for_status()
            if r.content[:2] == b"PK":
                with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                    root = ET.fromstring(z.read(z.namelist()[0]))
                for el in root.iter("list"):
                    sc = (el.findtext("stock_code") or "").strip()
                    cc = (el.findtext("corp_code") or "").strip()
                    nm = (el.findtext("corp_name") or "").strip()
                    if sc and cc:
                        mapping[sc.zfill(6)] = {"corp_code": cc, "corp_name": nm}
            else:
                err_txt = _parse_dart_error_xml(r.content)
                print(f"[DART 오류] corpCode.xml 다운로드 실패 ({err_txt})")
                print("  -> Colab 🔑(보안 비밀)의 DART_API_KEY 값/오타 또는 일일 한도(10,000건)를 확인하세요.")
        except Exception as e:
            print(f"[DART 오류] corpCode.xml 요청 실패 ({type(e).__name__}: {e})")

    # 폴백: 기존 Alpha Lab Drive 캐시(ckpt_corp_codes.json.gz)가 있으면 자동 재사용
    if not mapping:
        alt_p = Path(QUAL_CONFIG.get("ALPHA_LAB_CORP_CACHE") or "")
        if alt_p.exists() and alt_p.stat().st_size > 1000:
            try:
                raw_cc = json.loads(gzip.decompress(alt_p.read_bytes()).decode("utf-8"))
                for sc, cc in raw_cc.items():
                    if isinstance(cc, dict):
                        mapping[str(sc).zfill(6)] = {"corp_code": str(cc.get("corp_code") or ""), "corp_name": str(cc.get("corp_name") or sc)}
                    elif cc:
                        mapping[str(sc).zfill(6)] = {"corp_code": str(cc), "corp_name": str(sc).zfill(6)}
                print(f"[DART 매핑 폴백] 기존 Drive 캐시({alt_p.name})에서 {len(mapping):,}개사 로드")
            except Exception as e:
                print(f"[주의] Drive 폴백 캐시 로드 실패: {e}")

    if mapping:
        try:
            raw = gzip.compress(json.dumps(mapping, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
            (CACHE_DIR / "ckpt_corp_meta.json.gz").write_bytes(raw)
            if DRIVE_ROOT:
                (DRIVE_ROOT / "ckpt_corp_meta.json.gz").write_bytes(raw)
        except Exception:
            pass
    return mapping

CORP_META = load_corp_map()
print(f"[DART 매핑 완료] 상장사 {len(CORP_META):,}개사")

# 종목별 IR 업로드 하위 폴더 자동 생성 (예: ir_uploads/005380_현대차/)
if QUAL_CONFIG["CREATE_STOCK_SUBFOLDERS"]:
    for tk in TARGET_TICKERS:
        nm = (CORP_META.get(tk) or {}).get("corp_name") or tk
        safe_nm = re.sub(r"[^\w가-힣-]+", "", nm)
        (IR_UPLOAD_DIR / f"{tk}_{safe_nm}").mkdir(parents=True, exist_ok=True)
    print(f"[IR 폴더 준비 완료] {IR_UPLOAD_DIR} 아래 종목별 폴더 생성/확인됨")

# 핵심 정성 공시 선별 키워드 (우선순위별)
PRIORITY_PATTERNS = [
    (1, "주주환원/배당",   re.compile(r"현금·현물배당|배당결정|기업가치\s*제고|밸류업|자기주식.*(취득|소각)|주식소각|주주환원")),
    (1, "자본배분(신호⑦)", re.compile(r"유상증자|전환사채|신주인수권부사채|교환사채|합병|분할|영업양수|영업양도|타법인\s*주식.*취득|채무보증|자금대여")),
    (2, "IR/공정공시",     re.compile(r"기업설명회|공정공시|영업\(잠정\)실적|매출액또는손익구조")),
    (3, "정기보고서",      re.compile(r"^(사업보고서|반기보고서|분기보고서)")),
]
EXCLUDE_PAT = re.compile(r"증권발행실적보고서|일괄신고|소유상황보고서|임원ㆍ주요주주|주주총회소집공고|의결권대리행사")
_DART_ERR_LOGGED = set()

def clean_xml_to_text(raw_bytes: bytes) -> str:
    txt = None
    for enc in ("utf-8", "cp949", "euc-kr"):
        try:
            txt = raw_bytes.decode(enc)
            break
        except Exception:
            continue
    if txt is None:
        txt = raw_bytes.decode("utf-8", errors="ignore")
    txt = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", txt, flags=re.S | re.I)
    txt = re.sub(r"</(p|div|tr|li|h[1-6]|br)\s*>", "\n", txt, flags=re.I)
    txt = re.sub(r"<[^>]+>", " ", txt)
    txt = unescape(txt)
    txt = re.sub(r"[ \t\r\f\v]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt.strip()

def extract_focused_excerpts(text: str, is_periodic: bool, max_chars: int) -> str:
    """정기보고서는 배당·주주환원·자사주·사업위험 키워드 주변만 정밀 발췌, 단건 공시는 앞부분 전체 사용."""
    if not is_periodic or len(text) <= max_chars:
        return text[:max_chars]
    focus_kws = ["배당에 관한 사항", "배당정책", "주주환원", "자기주식", "소각", "기업가치 제고", "사업위험", "주요 위험", "유상증자", "합병", "분할"]
    chunks, used_spans = [], []
    for kw in focus_kws:
        for m in re.finditer(re.escape(kw), text):
            s = max(0, m.start() - 250)
            e = min(len(text), m.end() + 900)
            if any(not (e < us or s > ue) for us, ue in used_spans):
                continue
            used_spans.append((s, e))
            chunks.append(f"[발췌:{kw}] " + text[s:e].strip())
            if sum(len(c) for c in chunks) >= max_chars:
                return "\n---\n".join(chunks)[:max_chars]
            break
    return ("\n---\n".join(chunks) if chunks else text)[:max_chars]

def fetch_dart_qualitative_docs(corp_code: str, lookback_days: int, max_docs: int) -> list[dict]:
    if not DART_API_KEY or not corp_code:
        return []
    bgn = (today_dt - timedelta(days=lookback_days)).strftime("%Y%m%d")
    end = today_dt.strftime("%Y%m%d")
    try:
        r = requests.get(
            f"{DART_BASE}/list.json",
            params={"crtfc_key": DART_API_KEY, "corp_code": corp_code, "bgn_de": bgn, "end_de": end, "page_count": 100},
            headers=_HEADERS,
            timeout=25,
        )
        j = r.json()
        st = str(j.get("status") or "")
        if st != "000":
            if st not in ("013", "") and st not in _DART_ERR_LOGGED:
                _DART_ERR_LOGGED.add(st)
                print(f"\n  [DART 경고] list.json 응답 오류 (status={st}: {j.get('message', '')})", end=" ")
            return []
        items = j.get("list") or []
    except Exception as e:
        if "list_exc" not in _DART_ERR_LOGGED:
            _DART_ERR_LOGGED.add("list_exc")
            print(f"\n  [DART 경고] list.json 요청 실패 ({type(e).__name__})", end=" ")
        return []

    picked = []
    periodic_seen = 0
    for it in items:
        nm = str(it.get("report_nm") or "").strip()
        if EXCLUDE_PAT.search(nm):
            continue
        matched_cat = None
        prio = 99
        for pr, cat, rx in PRIORITY_PATTERNS:
            if rx.search(nm):
                matched_cat, prio = cat, pr
                break
        if matched_cat is None:
            continue
        if matched_cat == "정기보고서":
            if periodic_seen >= 2:
                continue
            periodic_seen += 1
        picked.append((prio, str(it.get("rcept_dt") or ""), matched_cat, nm, str(it.get("rcept_no") or "")))

    # 우선순위 오름차순 -> 최신 날짜 내림차순
    picked.sort(key=lambda x: (x[0], -int(re.sub(r"\D", "", x[1]) or "0")))
    picked = picked[:max_docs]

    docs = []
    for _, rdt, cat, nm, rno in picked:
        if not rno:
            continue
        text = ""
        try:
            dr = requests.get(f"{DART_BASE}/document.xml", params={"crtfc_key": DART_API_KEY, "rcept_no": rno}, headers=_HEADERS, timeout=30)
            if dr.status_code == 200 and dr.content[:2] == b"PK":
                with zipfile.ZipFile(io.BytesIO(dr.content)) as z:
                    all_names = z.namelist()
                    doc_names = [fn for fn in all_names if fn.lower().endswith((".xml", ".htm", ".html"))] or all_names
                    xml_names = sorted(doc_names, key=lambda fn: z.getinfo(fn).file_size, reverse=True)
                    if xml_names:
                        raw = z.read(xml_names[0])
                        text = extract_focused_excerpts(clean_xml_to_text(raw), cat == "정기보고서", QUAL_CONFIG["MAX_CHARS_PER_DOC"])
            time.sleep(0.12)
        except Exception:
            text = ""
        docs.append({
            "source_type": "DART공시",
            "category": cat,
            "date": f"{rdt[:4]}-{rdt[4:6]}-{rdt[6:8]}" if len(rdt) == 8 else rdt,
            "title": nm,
            "ref": rno,
            "url": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rno}",
            "text": text or f"(원문 자동 추출 실패 — 제목: {nm})",
        })
    return docs

def extract_pdf_text(pdf_bytes: bytes, max_pages: int = 25, max_chars: int = 8000) -> str:
    if PdfReader is None:
        return "(pypdf 미설치로 PDF 텍스트 추출 생략)"
    try:
        reader = PdfReader(io.BytesIO(pdf_bytes))
        parts = []
        for idx, page in enumerate(reader.pages[:max_pages]):
            t = (page.extract_text() or "").strip()
            if t:
                parts.append(f"[p.{idx+1}] {t}")
            if sum(len(x) for x in parts) >= max_chars:
                break
        out = "\n".join(parts)
        return out[:max_chars] if out else "(이미지 위주 PDF — 추출 가능한 텍스트 부족)"
    except Exception as e:
        return f"(PDF 파싱 오류: {type(e).__name__})"

# 합법 공개 공식 IR PDF 자동 수집 (robots.txt 준수, 검증된 2개사: 삼성전자·KB금융)
PUBLIC_IR_SOURCES = {
    "005930": {
        "list_url": "https://www.samsung.com/sec/ir/main/",
        "pattern": r'href="(?P<u>(?:https:)?//images\.samsung\.com/kdp/ir/events/20\d{2}/20\d{2}_[1-4]Q_conference_kor\.pdf)"',
    },
    "105560": {
        "list_url": "https://www.kbfg.com/kor/ir/mgt-performance/list.jsp",
        "pattern": r'href="(?P<u>/common/jsp/fileDownUtil\.jsp\?filepath=/data/ir_presentation/20\d{2}_(?:1Q|1H|3Q|4Q)PT_KOR\.pdf)"',
    },
}

def _robots_allowed(url: str) -> bool:
    try:
        p = urlparse(url)
        rr = requests.get(f"{p.scheme}://{p.netloc}/robots.txt", headers=_HEADERS, timeout=8)
        if rr.status_code != 200:
            return True
        rp = RobotFileParser()
        rp.parse(rr.text.splitlines())
        return rp.can_fetch("*", url)
    except Exception:
        return True

def fetch_public_ir_pdf(ticker: str) -> list[dict]:
    if not QUAL_CONFIG["AUTO_FETCH_PUBLIC_IR"] or ticker not in PUBLIC_IR_SOURCES:
        return []
    cfg = PUBLIC_IR_SOURCES[ticker]
    list_url = cfg["list_url"]
    try:
        if not _robots_allowed(list_url):
            return []
        r = requests.get(list_url, headers=_HEADERS, timeout=15)
        if r.status_code != 200:
            return []
        matches = list(re.finditer(cfg["pattern"], r.text))
        if not matches:
            return []
        raw_u = matches[0].group("u")
        if raw_u.startswith("//"):
            file_url = "https:" + raw_u
        else:
            file_url = urljoin(list_url, raw_u)
        if not _robots_allowed(file_url):
            return []
        fr = requests.get(file_url, headers=_HEADERS, timeout=25)
        if fr.status_code != 200 or len(fr.content) < 1000:
            return []
        txt = extract_pdf_text(fr.content, max_pages=25, max_chars=QUAL_CONFIG["MAX_CHARS_PER_DOC"])
        return [{
            "source_type": "공식IR(자동)",
            "category": "실적발표IR",
            "date": as_of_str,
            "title": Path(urlparse(file_url).path).name or "공식 IR 발표자료",
            "ref": file_url,
            "url": file_url,
            "text": txt,
        }]
    except Exception:
        return []

def load_drive_uploaded_ir(ticker: str, corp_name: str) -> list[dict]:
    """MyDrive/.../ir_uploads/<ticker>_.../ 폴더 또는 파일명에 티커/종목명이 포함된 PDF·TXT·MD 자동 로드."""
    docs = []
    if not IR_UPLOAD_DIR.exists():
        return docs
    cand_files = []
    # 1) 종목 전용 하위 폴더 안의 모든 파일
    for sub in IR_UPLOAD_DIR.iterdir():
        if sub.is_dir() and (ticker in sub.name or (corp_name and corp_name in sub.name)):
            for fp in sorted(sub.glob("*")):
                if fp.is_file() and fp.suffix.lower() in (".pdf", ".txt", ".md"):
                    cand_files.append(fp)
    # 2) 루트 ir_uploads/ 에 직접 올린 파일 (파일명에 티커 또는 종목명 포함)
    for fp in sorted(IR_UPLOAD_DIR.glob("*")):
        if fp.is_file() and fp.suffix.lower() in (".pdf", ".txt", ".md") and fp.name != "README_IR_업로드_안내.txt":
            if ticker in fp.name or (corp_name and corp_name in fp.name):
                cand_files.append(fp)

    for fp in cand_files[:5]:
        try:
            if fp.suffix.lower() == ".pdf":
                txt = extract_pdf_text(fp.read_bytes(), max_pages=30, max_chars=QUAL_CONFIG["MAX_CHARS_PER_DOC"])
            else:
                txt = fp.read_text(encoding="utf-8", errors="ignore")[:QUAL_CONFIG["MAX_CHARS_PER_DOC"]]
            docs.append({
                "source_type": "Drive업로드IR",
                "category": "사용자IR/리포트",
                "date": datetime.fromtimestamp(fp.stat().st_mtime, tz=KST).strftime("%Y-%m-%d"),
                "title": fp.name,
                "ref": fp.name,
                "url": "",
                "text": txt,
            })
        except Exception:
            continue
    return docs

print("[수집기 준비 완료] OpenDART 원문 + 공식 공개 IR(robots.txt 준수) + Drive 업로드 폴더")

In [ ]:
# ==============================================================================
# 4. 종목별 문서 수집 + Google AI Studio Gemini 정성 심사 (신호 ⑤ · ⑥ · ⑦)
# ==============================================================================
SYSTEM_PROMPT = """너는 한국 상장 배당성장주(10년 장기보유) 전문 심사역이다.
반드시 아래 규칙을 엄격히 지켜 JSON만 출력하라:
1. 제공된 DART 공시 원문과 IR 자료에 실제로 적힌 내용만 근거로 판단하라. 원문에 없는 수치·사실을 절대 추측하거나 지어내지 마라.
2. 원문에 근거가 없으면 반드시 status를 "확인불가" 또는 "없음"으로 두고 evidence_quote에 "제공된 공시·IR 원문에서 해당 내용 미확인"이라고 적어라.
3. 모든 핵심 판단(신호 ⑤⑥⑦)에는 원문 문장을 가능한 그대로 인용하고 출처(rcept_no 또는 파일명)를 함께 적어라.

[심사 기준]
- 신호 ⑤ 배당 변동 사유: 최근 1년 정기배당 삭감·중단·감액 공시가 있었는지, 있었다면 사유가 업황성(실적 악화) / 재배치성(자사주 소각 전환) / 규제성(당국 권고) 중 무엇인지 분류.
- 신호 ⑥ 주주환원 정책: 기업가치 제고 계획(밸류업), 배당성향 목표, 자사주 매입·소각 정책이 기존 대비 상향 / 유지 / 후퇴 중 어디인지 판단.
- 신호 ⑦ 자본배분 비효율: 순이익 1년치 수준의 대형 M&A, 계열사 지원·합병, 물적분할 후 자회사 상장, 발행주식 5% 이상 대규모 유상증자·CB/BW 발행 등 주주가치 희석 징후가 있는지 감시.
- 종합 정성 등급:
  - "재검토": 업황성 배당 삭감·중단, 주주환원 정책 공식 철회/후퇴, 또는 심각한 신호 ⑦ 경고 발생 시
  - "관찰": 확인이 필요한 리스크 징후(주의)가 1건 이상 있거나, 재배치성/규제성 배당 변화 확인 필요 시
  - "유지": 중대한 정성 리스크 징후 없음

반드시 아래 JSON 스키마로만 응답하라:
{
  "overall_grade": "유지|관찰|재검토",
  "signal_5_div_cut": {"status": "없음|업황성|재배치성|규제성|확인불가", "summary": "한 줄 요약", "evidence_quote": "원문 인용", "source_ref": "접수번호 또는 파일명"},
  "signal_6_return_policy": {"status": "상향|유지|후퇴|확인불가", "summary": "한 줄 요약", "evidence_quote": "원문 인용", "source_ref": "접수번호 또는 파일명"},
  "signal_7_capital_alloc": {"status": "정상|주의|경고", "summary": "한 줄 요약", "evidence_quote": "원문 인용", "source_ref": "접수번호 또는 파일명"},
  "ir_key_points": ["핵심 포인트 1", "핵심 포인트 2"],
  "analyst_comment": "10년 배당성장 투자자 관점의 2~3문장 종합 코멘트"
}
"""

def _norm_sig(val, default_status="확인불가") -> dict:
    if isinstance(val, dict):
        return {
            "status": str(val.get("status") or default_status),
            "summary": str(val.get("summary") or ""),
            "evidence_quote": str(val.get("evidence_quote") or ""),
            "source_ref": str(val.get("source_ref") or ""),
        }
    return {"status": str(val or default_status), "summary": "", "evidence_quote": "", "source_ref": ""}

def _extract_gemini_json(resp_json: dict) -> dict:
    cands = resp_json.get("candidates") or []
    if not cands:
        fb = resp_json.get("promptFeedback") or {}
        raise ValueError(f"candidates 없음 (blockReason={fb.get('blockReason', 'unknown')})")
    content = (cands[0].get("content") or {})
    cparts = content.get("parts") or []
    # Gemini 2.5 thinking 파트(thought=True)를 제외하고 최종 JSON 텍스트만 추출
    non_thought = [p.get("text", "") for p in cparts if isinstance(p, dict) and not p.get("thought") and p.get("text")]
    all_txt = non_thought or [p.get("text", "") for p in cparts if isinstance(p, dict) and p.get("text")]
    raw_txt = "\n".join(all_txt).strip()
    if not raw_txt:
        raise ValueError(f"응답 텍스트 비어있음 (finishReason={cands[0].get('finishReason', 'unknown')})")
    raw_txt = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_txt, flags=re.S | re.I).strip()
    try:
        parsed = json.loads(raw_txt)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw_txt, flags=re.S)
        if not m:
            raise
        parsed = json.loads(m.group(0))
    if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
        parsed = parsed[0]
    if not isinstance(parsed, dict):
        raise ValueError(f"JSON 객체 아님 ({type(parsed).__name__})")
    pts = parsed.get("ir_key_points")
    if isinstance(pts, str):
        pts = [pts]
    elif not isinstance(pts, list):
        pts = []
    return {
        "overall_grade": str(parsed.get("overall_grade") or "관찰"),
        "signal_5_div_cut": _norm_sig(parsed.get("signal_5_div_cut"), "확인불가"),
        "signal_6_return_policy": _norm_sig(parsed.get("signal_6_return_policy"), "확인불가"),
        "signal_7_capital_alloc": _norm_sig(parsed.get("signal_7_capital_alloc"), "정상"),
        "ir_key_points": [str(x) for x in pts if x],
        "analyst_comment": str(parsed.get("analyst_comment") or ""),
    }

def call_gemini_qualitative(corp_name: str, ticker: str, docs: list[dict]) -> dict:
    if not GEMINI_API_KEY:
        return {
            "overall_grade": "원문발췌(Dry-Run)",
            "signal_5_div_cut": {"status": "미실행", "summary": "GEMINI_API_KEY 미설정 — 공시 원문 수집만 완료", "evidence_quote": "", "source_ref": ""},
            "signal_6_return_policy": {"status": "미실행", "summary": f"수집 문서 {len(docs)}건", "evidence_quote": "", "source_ref": ""},
            "signal_7_capital_alloc": {"status": "미실행", "summary": "Colab Secrets에 GEMINI_API_KEY 등록 시 자동 판독", "evidence_quote": "", "source_ref": ""},
            "ir_key_points": [f"[{d['source_type']}] {d['date']} {d['title']}" for d in docs[:4]],
            "analyst_comment": "GEMINI_API_KEY를 Colab Secrets에 등록하고 [노트북 액세스]를 ON으로 켠 뒤 재실행하면 정성 리스크(⑤⑥⑦) AI 판독이 자동 수행됩니다.",
            "_model": "dry-run",
        }

    # 컨텍스트 조립 (토큰 상한 준수)
    parts = [f"대상 기업: {corp_name} ({ticker}) | 분석 기준일: {as_of_str}\n"]
    used = 0
    for idx, d in enumerate(docs, 1):
        block = f"\n=== 문서 {idx} | [{d['source_type']}/{d['category']}] {d['date']} | 제목: {d['title']} | 출처ID: {d['ref']} ===\n{d['text']}\n"
        if used + len(block) > QUAL_CONFIG["MAX_TOTAL_CONTEXT_CHARS"]:
            rem = QUAL_CONFIG["MAX_TOTAL_CONTEXT_CHARS"] - used
            if rem > 500:
                parts.append(block[:rem])
            break
        parts.append(block)
        used += len(block)
    user_content = "".join(parts)

    payload = {
        "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
        "contents": [{"role": "user", "parts": [{"text": user_content}]}],
        "generationConfig": {"responseMimeType": "application/json"},
    }

    last_err = ""
    for model_name in QUAL_CONFIG["GEMINI_MODELS"]:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent"
        try:
            r = requests.post(url, params={"key": GEMINI_API_KEY}, json=payload, timeout=90)
            for wait_s in (10.0, 20.0):
                if r.status_code != 429:
                    break
                time.sleep(wait_s)
                r = requests.post(url, params={"key": GEMINI_API_KEY}, json=payload, timeout=90)
            if r.status_code != 200:
                last_err = f"{model_name} HTTP {r.status_code}: {r.text[:160]}"
                continue
            parsed = _extract_gemini_json(r.json())
            parsed["_model"] = model_name
            return parsed
        except Exception as e:
            last_err = f"{model_name} {type(e).__name__}: {e}"
            continue

    return {
        "overall_grade": "분석오류",
        "signal_5_div_cut": {"status": "확인불가", "summary": last_err[:120], "evidence_quote": "", "source_ref": ""},
        "signal_6_return_policy": {"status": "확인불가", "summary": "", "evidence_quote": "", "source_ref": ""},
        "signal_7_capital_alloc": {"status": "주의", "summary": "", "evidence_quote": "", "source_ref": ""},
        "ir_key_points": [],
        "analyst_comment": f"Gemini API 호출 실패 ({last_err[:160]})",
        "_model": "error",
    }

# 월 단위 이어받기 체크포인트 로드
CKPT_QUAL_NAME = f"ckpt_qual_ai_{as_of_str[:7]}.json.gz"
ckpt_path_drive = (DRIVE_ROOT / CKPT_QUAL_NAME) if DRIVE_ROOT else None
ckpt_path_local = CACHE_DIR / CKPT_QUAL_NAME

RESULTS_MAP = {}
if not QUAL_CONFIG["FORCE_REANALYZE"]:
    for cp in [ckpt_path_drive, ckpt_path_local]:
        if cp and cp.exists() and cp.stat().st_size > 20:
            try:
                RESULTS_MAP = json.loads(gzip.decompress(cp.read_bytes()).decode("utf-8"))
                break
            except Exception:
                pass

def save_qual_ckpt():
    raw = gzip.compress(json.dumps(RESULTS_MAP, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
    ckpt_path_local.write_bytes(raw)
    if ckpt_path_drive:
        try:
            ckpt_path_drive.write_bytes(raw)
        except Exception:
            pass

RAW_DOCS_BUNDLE = {}
_GEMINI_BILLING_STOPPED = False

for idx, tk in enumerate(TARGET_TICKERS, 1):
    meta = CORP_META.get(tk) or {"corp_code": "", "corp_name": tk}
    cname, ccode = meta["corp_name"], meta["corp_code"]
    if tk in RESULTS_MAP and isinstance(RESULTS_MAP[tk], dict) and RESULTS_MAP[tk].get("ai", {}).get("_model") not in (None, "error", "dry-run", "bundle-only"):
        print(f"  [{idx}/{len(TARGET_TICKERS)}] {cname}({tk}) — 이번 달 분석 캐시 재사용")
        continue

    print(f"  [{idx}/{len(TARGET_TICKERS)}] {cname}({tk}) 수집 중...", end=" ")
    dart_docs = fetch_dart_qualitative_docs(ccode, QUAL_CONFIG["LOOKBACK_DAYS"], QUAL_CONFIG["MAX_DISCLOSURES_PER_STOCK"])
    pub_ir = fetch_public_ir_pdf(tk)
    drv_ir = load_drive_uploaded_ir(tk, cname)
    all_docs = drv_ir + pub_ir + dart_docs
    RAW_DOCS_BUNDLE[tk] = {
        "ticker": tk,
        "name": cname,
        "role": TARGET_ROLE.get(tk, "관심"),
        "as_of": as_of_str,
        "docs": all_docs,
    }
    print(f"DART {len(dart_docs)}건 · 공식IR {len(pub_ir)}건 · Drive업로드 {len(drv_ir)}건", end=" ")

    if all_docs and not _GEMINI_BILLING_STOPPED:
        print("-> Gemini 심사...", end=" ")
        ai_res = call_gemini_qualitative(cname, tk, all_docs)
        if ai_res.get("_model") == "error" and "HTTP 402" in str(ai_res.get("analyst_comment") or ""):
            _GEMINI_BILLING_STOPPED = True
            print("\n  [안내] Gemini API HTTP 402(선불 크레딧 잠금) 감지 -> 나머지 종목은 API 호출을 생략하고 공시·IR 원문 번들만 고속 수집합니다.")
    elif all_docs and _GEMINI_BILLING_STOPPED:
        ai_res = {
            "overall_grade": "원문수집완료",
            "signal_5_div_cut": {"status": "원문저장", "summary": "Drive raw_bundle_latest.md 저장 완료 (무료 심사 가능)", "evidence_quote": "", "source_ref": ""},
            "signal_6_return_policy": {"status": "원문저장", "summary": f"수집 문서 {len(all_docs)}건", "evidence_quote": "", "source_ref": ""},
            "signal_7_capital_alloc": {"status": "원문저장", "summary": "AI 채팅/Antigravity에서 Drive 원문 번들 바로 판독 가능", "evidence_quote": "", "source_ref": ""},
            "ir_key_points": [f"[{d['source_type']}] {d['date']} {d['title']}" for d in all_docs[:4]],
            "analyst_comment": "Gemini API 402 우회: DART 공시 원문과 IR PDF를 Drive 번들로 저장했습니다. Antigravity 채팅에서 바로 무료 정성 심사가 가능합니다.",
            "_model": "bundle-only",
        }
    else:
        ai_res = {
            "overall_grade": "자료없음",
            "signal_5_div_cut": {"status": "확인불가", "summary": "최근 1년 핵심 공시·IR 문서 없음", "evidence_quote": "", "source_ref": ""},
            "signal_6_return_policy": {"status": "확인불가", "summary": "", "evidence_quote": "", "source_ref": ""},
            "signal_7_capital_alloc": {"status": "정상", "summary": "", "evidence_quote": "", "source_ref": ""},
            "ir_key_points": [],
            "analyst_comment": "수집된 공시나 업로드 IR 파일이 없습니다.",
            "_model": "none",
        }
    print(f"완료 ({ai_res.get('overall_grade')})")

    RESULTS_MAP[tk] = {
        "ticker": tk,
        "name": cname,
        "role": TARGET_ROLE.get(tk, "관심"),
        "n_dart": len(dart_docs),
        "n_pub_ir": len(pub_ir),
        "n_drv_ir": len(drv_ir),
        "docs_meta": [{k: d[k] for k in ("source_type", "category", "date", "title", "ref", "url")} for d in all_docs],
        "ai": ai_res,
    }
    save_qual_ckpt()
    if GEMINI_API_KEY and not _GEMINI_BILLING_STOPPED and idx < len(TARGET_TICKERS):
        time.sleep(QUAL_CONFIG["GEMINI_SLEEP_SEC"])

# 수집된 원문 전체 번들(.json.gz + .md) 자동 저장 -> API 크레딧 없이도 무료 AI 심사 가능!
if RAW_DOCS_BUNDLE:
    bundle_gz = gzip.compress(json.dumps(RAW_DOCS_BUNDLE, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
    (CACHE_DIR / "qualitative_raw_bundle_latest.json.gz").write_bytes(bundle_gz)
    md_lines = [
        f"# OpenDART 공시 원문 + IR PDF 정성 심사 원문 번들 ({as_of_str})",
        f"- 대상 종목: {len(RAW_DOCS_BUNDLE)}종목 | 최근 {QUAL_CONFIG['LOOKBACK_DAYS']}일 핵심 공시 + IR PDF 발췌",
        "- 심사 항목: 신호 ⑤(배당 변동 사유) · 신호 ⑥(주주환원 정책) · 신호 ⑦(자본배분 비효율)",
        "",
    ]
    for tk in TARGET_TICKERS:
        b = RAW_DOCS_BUNDLE.get(tk)
        if not b:
            continue
        md_lines.append(f"## [{b['role']}] {b['name']} ({tk}) — 문서 {len(b['docs'])}건")
        for di, d in enumerate(b["docs"], 1):
            md_lines.append(f"### {di}. [{d['source_type']}/{d['category']}] {d['date']} | {d['title']} (출처: {d['ref']})")
            if d.get("url"):
                md_lines.append(f"- 원문 링크: {d['url']}")
            md_lines.append(d["text"][:4500])
            md_lines.append("")
    bundle_md_txt = "\n".join(md_lines)
    Path("qualitative_raw_bundle_latest.md").write_text(bundle_md_txt, encoding="utf-8")
    if DRIVE_REPORT_DIR is not None:
        try:
            DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
            (DRIVE_REPORT_DIR / "qualitative_raw_bundle_latest.json.gz").write_bytes(bundle_gz)
            (DRIVE_REPORT_DIR / "qualitative_raw_bundle_latest.md").write_text(bundle_md_txt, encoding="utf-8")
            print(f"[원문 번들 Drive 저장 완료] {DRIVE_REPORT_DIR / 'qualitative_raw_bundle_latest.md'} ({len(bundle_md_txt)/1024:.1f} KB)")
        except Exception as e:
            print(f"[주의] 원문 번들 Drive 저장 실패: {e}")

ORDERED_RESULTS = [RESULTS_MAP[tk] for tk in TARGET_TICKERS if tk in RESULTS_MAP]
SUMMARY_QUAL_DF = pd.DataFrame([
    {
        "구분": r["role"],
        "종목": f"{r['name']}({r['ticker']})",
        "정성 등급": (r.get("ai") or {}).get("overall_grade", "-"),
        "신호⑤ 배당변동": _norm_sig((r.get("ai") or {}).get("signal_5_div_cut"), "-").get("status", "-"),
        "신호⑥ 환원정책": _norm_sig((r.get("ai") or {}).get("signal_6_return_policy"), "-").get("status", "-"),
        "신호⑦ 자본배분": _norm_sig((r.get("ai") or {}).get("signal_7_capital_alloc"), "-").get("status", "-"),
        "분석 문서": f"DART {r['n_dart']} · 공식IR {r['n_pub_ir']} · 업로드 {r['n_drv_ir']}",
        "심사역 한줄 요약": str((r.get("ai") or {}).get("analyst_comment") or "")[:90],
    }
    for r in ORDERED_RESULTS
])

print(f"\n■ [OpenDART + IR × Gemini AI 정성 리스크 감시 요약 ({as_of_str})]")
display(SUMMARY_QUAL_DF)

In [ ]:
# ==============================================================================
# 5. 종목별 정성 리스크 심사 카드 A4 PDF/HTML 리포트 생성 & Drive 자동 저장
# ==============================================================================
def pill_html(val: str) -> str:
    s = str(val or "-")
    if s in ("유지", "상향", "없음", "정상"):
        cls = "p-green"
    elif s in ("관찰", "주의", "재배치성", "규제성", "확인불가"):
        cls = "p-amber"
    elif s in ("재검토", "후퇴", "경고", "업황성"):
        cls = "p-red"
    else:
        cls = "p-gray"
    return f'<span class="pill {cls}">{escape(s)}</span>'

def _as_sig_dict(v) -> dict:
    return v if isinstance(v, dict) else {"status": str(v or "-"), "summary": "", "evidence_quote": "", "source_ref": ""}

sum_rows_html = []
for r in ORDERED_RESULTS:
    ai = r.get("ai") or {}
    s5 = _as_sig_dict(ai.get("signal_5_div_cut")).get("status", "-")
    s6 = _as_sig_dict(ai.get("signal_6_return_policy")).get("status", "-")
    s7 = _as_sig_dict(ai.get("signal_7_capital_alloc")).get("status", "-")
    sum_rows_html.append(
        f"<tr><td class='col-left'>{escape(r['role'])}</td>"
        f"<td class='col-left'>{escape(r['name'])} ({escape(r['ticker'])})</td>"
        f"<td>{pill_html(ai.get('overall_grade'))}</td>"
        f"<td>{pill_html(s5)}</td><td>{pill_html(s6)}</td><td>{pill_html(s7)}</td>"
        f"<td>DART {r['n_dart']} · IR {r['n_pub_ir']+r['n_drv_ir']}</td></tr>"
    )

detail_cards = []
for r in ORDERED_RESULTS:
    ai = r.get("ai") or {}
    s5 = _as_sig_dict(ai.get("signal_5_div_cut"))
    s6 = _as_sig_dict(ai.get("signal_6_return_policy"))
    s7 = _as_sig_dict(ai.get("signal_7_capital_alloc"))
    raw_pts = ai.get("ir_key_points") or []
    if isinstance(raw_pts, str):
        raw_pts = [raw_pts]
    pts = "".join(f"<li>{escape(str(x))}</li>" for x in raw_pts)
    pts_block = f"<div class='subsec'><b>핵심 IR · 공시 포인트</b><ul>{pts}</ul></div>" if pts else ""
    doc_links = []
    for dm in (r.get("docs_meta") or [])[:8]:
        label = f"[{dm['source_type']}] {dm['date']} {dm['title']}"
        if dm.get("url"):
            u_esc = escape(str(dm["url"]))
            ref_esc = escape(str(dm.get("ref") or ""))
            doc_links.append(f'<li><a href="{u_esc}" target="_blank">{escape(label)}</a> (출처: {ref_esc})</li>')
        else:
            doc_links.append(f"<li>{escape(label)}</li>")
    doc_links_html = "".join(doc_links) or "<li>수집 문서 없음</li>"

    def sig_box(title, d):
        q = escape(str(d.get("evidence_quote") or ""))
        ref = escape(str(d.get("source_ref") or ""))
        quote_html = f"<div class='quote'>“{q}” <span class='ref'>({ref})</span></div>" if q else ""
        return (
            f"<div class='sig-box'><div class='sig-head'><b>{title}</b> {pill_html(d.get('status'))}</div>"
            f"<div class='sig-sum'>{escape(str(d.get('summary') or '-'))}</div>"
            f"{quote_html}</div>"
        )

    role_esc = escape(str(r.get("role") or ""))
    name_esc = escape(str(r.get("name") or ""))
    tk_esc = escape(str(r.get("ticker") or ""))
    grade_pill = pill_html(ai.get("overall_grade"))
    model_esc = escape(str(ai.get("_model", "-")))
    comment_esc = escape(str(ai.get("analyst_comment") or ""))
    sb5 = sig_box("신호 ⑤ 배당 변동 사유", s5)
    sb6 = sig_box("신호 ⑥ 주주환원 정책", s6)
    sb7 = sig_box("신호 ⑦ 자본배분 비효율", s7)
    n_docs = len(r.get("docs_meta") or [])

    detail_cards.append(f"""
    <div class="card stock-card">
      <div class="sc-head">
        <div><span class="role-tag">{role_esc}</span> <b>{name_esc}</b> ({tk_esc})</div>
        <div>종합 정성 등급: {grade_pill} <span class="ref">model: {model_esc}</span></div>
      </div>
      <div class="comment">{comment_esc}</div>
      <div class="sig-grid">
        {sb5}
        {sb6}
        {sb7}
      </div>
      {pts_block}
      <div class="subsec"><b>분석 근거 문서 ({n_docs}건)</b><ul class="doc-list">{doc_links_html}</ul></div>
    </div>
    """)

sum_rows_str = "".join(sum_rows_html)
detail_cards_str = "".join(detail_cards)
lookback_days = QUAL_CONFIG["LOOKBACK_DAYS"]
n_targets = len(ORDERED_RESULTS)

html_report = f"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><title>OpenDART+IR 정성 리스크 AI 감시 리포트 ({as_of_str})</title>
<style>
  @page {{ size: A4 portrait; margin: 11mm; }}
  body {{ margin:0; padding:12px; background:#F8FAFC; color:#0F172A; font-family:'NanumGothic',sans-serif; font-size:10.3px; line-height:1.5; }}
  .wrap {{ max-width:960px; margin:0 auto; }}
  .head {{ background:linear-gradient(135deg,#0F2922,#134E4A); color:#fff; padding:16px 20px; border-radius:10px; margin-bottom:12px; border-bottom:4px solid #D4A34A; }}
  .card {{ background:#fff; border:1px solid #E2E8F0; border-radius:10px; padding:12px 14px; margin-bottom:10px; break-inside:avoid; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.3px; }}
  table.tbl th {{ background:#F1F5F9; padding:5.5px; border-top:1.5px solid #0F2922; border-bottom:1px solid #CBD5E1; text-align:center; }}
  table.tbl td {{ padding:5.5px; border-bottom:1px solid #EEF2F6; text-align:center; }}
  table.tbl td.col-left, table.tbl th:first-child, table.tbl th:nth-child(2) {{ text-align:left; font-weight:700; }}
  .pill {{ display:inline-block; padding:1.5px 7px; border-radius:999px; font-size:8.5px; font-weight:700; white-space:nowrap; }}
  .p-green {{ background:#DCFCE7; color:#166534; }} .p-amber {{ background:#FEF3C7; color:#92400E; }}
  .p-red {{ background:#FEE2E2; color:#991B1B; }} .p-gray {{ background:#E2E8F0; color:#475569; }}
  .sc-head {{ display:flex; justify-content:space-between; align-items:center; border-bottom:1px solid #E2E8F0; padding-bottom:6px; margin-bottom:7px; font-size:12px; }}
  .role-tag {{ background:#0F2922; color:#fff; padding:1px 6px; border-radius:4px; font-size:8.5px; margin-right:4px; }}
  .comment {{ background:#F0FDF9; border-left:3px solid #0F766E; padding:7px 10px; border-radius:5px; margin-bottom:8px; color:#064E3B; }}
  .sig-grid {{ display:grid; grid-template-columns:repeat(3,1fr); gap:8px; margin-bottom:8px; }}
  .sig-box {{ background:#F8FAFC; border:1px solid #E2E8F0; border-radius:7px; padding:7px 8px; }}
  .sig-head {{ display:flex; justify-content:space-between; align-items:center; margin-bottom:4px; font-size:9.4px; }}
  .sig-sum {{ font-size:9.2px; color:#1E293B; }}
  .quote {{ margin-top:4px; font-size:8.4px; color:#475569; font-style:italic; border-top:1px dashed #CBD5E1; padding-top:4px; }}
  .ref {{ font-size:8px; color:#64748B; font-style:normal; }}
  .subsec {{ margin-top:6px; font-size:9.2px; }} .subsec ul {{ margin:3px 0 0 16px; padding:0; }}
  .doc-list a {{ color:#0F766E; text-decoration:none; }}
</style></head><body><div class="wrap">
  <div class="head">
    <h1 style="margin:0 0 4px;font-size:17px;">OpenDART 공시 원문 + IR PDF × Gemini AI 정성 리스크 리포트 ({as_of_str})</h1>
    <div>감시 대상 {n_targets}종목 · 최근 {lookback_days}일 공시 + Drive 업로드 IR PDF 통합 심사 (신호 ⑤·⑥·⑦)</div>
  </div>
  <div class="card">
    <h2 style="margin:0 0 8px;font-size:12.5px;color:#0F2922;">01. 전체 종목 정성 리스크 신호등 요약</h2>
    <table class="tbl"><thead><tr><th>구분</th><th>종목</th><th>종합 등급</th><th>신호⑤ 배당변동</th><th>신호⑥ 환원정책</th><th>신호⑦ 자본배분</th><th>분석 문서</th></tr></thead>
    <tbody>{sum_rows_str}</tbody></table>
  </div>
  {detail_cards_str}
</div></body></html>"""

r_html = f"qualitative_ai_report_{run_stamp}.html"
r_pdf = f"qualitative_ai_report_{run_stamp}.pdf"
Path(r_html).write_text(html_report, encoding="utf-8")

pdf_ok = False
try:
    from weasyprint import HTML as WP_HTML
    WP_HTML(string=html_report).write_pdf(r_pdf)
    pdf_ok = True
except Exception as e:
    print(f"[안내] PDF 변환 생략({type(e).__name__}: {e}) -> HTML 리포트는 정상 저장됩니다.")

print(f"[리포트 완료] HTML={Path(r_html).stat().st_size/1024:.1f} KB | PDF={'성공' if pdf_ok else 'HTML 저장 완료 (Colab에서 PDF 자동 생성)'}")
# Google Drive 자동 연결 재시도 (Cell 2에서 미연결/취소된 경우에도 Cell 5에서 다시 시도)
if IN_COLAB and QUAL_CONFIG.get("USE_GOOGLE_DRIVE", True) and DRIVE_REPORT_DIR is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(QUAL_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 재연결 완료] {DRIVE_REPORT_DIR}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬(/content/{r_pdf if pdf_ok else r_html})에만 저장됨 ({type(e).__name__}: {e})")

if DRIVE_REPORT_DIR is not None:
    try:
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        (DRIVE_REPORT_DIR / r_html).write_text(html_report, encoding="utf-8")
        (DRIVE_REPORT_DIR / "qualitative_ai_report_latest.html").write_text(html_report, encoding="utf-8")
        saved_targets = [DRIVE_REPORT_DIR / r_html]
        if pdf_ok and Path(r_pdf).exists():
            pdf_bytes = Path(r_pdf).read_bytes()
            (DRIVE_REPORT_DIR / r_pdf).write_bytes(pdf_bytes)
            (DRIVE_REPORT_DIR / "qualitative_ai_report_latest.pdf").write_bytes(pdf_bytes)
            saved_targets.insert(0, DRIVE_REPORT_DIR / r_pdf)
        csv_name = f"qualitative_summary_{run_stamp}.csv"
        SUMMARY_QUAL_DF.to_csv(DRIVE_REPORT_DIR / csv_name, index=False, encoding="utf-8-sig")
        SUMMARY_QUAL_DF.to_csv(DRIVE_REPORT_DIR / "qualitative_summary_latest.csv", index=False, encoding="utf-8-sig")
        saved_targets.append(DRIVE_REPORT_DIR / csv_name)
        print(f"[Drive 저장 검증 완료] 폴더: {DRIVE_REPORT_DIR}")
        for fp in saved_targets:
            print(f"  - {fp.name} ({fp.stat().st_size/1024:.1f} KB)")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")